# RENER

In [ ]:
%load_ext autoreload 
%autoreload 2

In [ ]:
import json

# Load jsonl file to use as data to be retrieved
with open("../data/labelled/all_parallel.jsonl") as test_f:
    data = [json.loads(line) for line in test_f]

In [ ]:
from glob import glob

# Load input
doc_files = glob("../data/raw/doc2phrases_dataset/**")

input_data = []
for f in doc_files:
    with open(f) as doc_f:
        for i, line in enumerate(doc_f.readlines()):
            input_data.append(
                {
                    "id": i,
                    "text": line,
                    "pred": "",
                    "annotations": [],
                }
            )

In [ ]:
from time import time
from util import extract_ner_labels
from tqdm.notebook import tqdm

from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.dataclasses import Document
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever
from haystack_integrations.components.generators.ollama import OllamaGenerator
from haystack.components.rankers import SentenceTransformersSimilarityRanker

from prompts import guidelines_llama3_legal_lite, default_examples, guidelines_en
from datetime import datetime

# Create document store
document_store = InMemoryDocumentStore()
docs = [Document(content=d["input"], meta={"output": d["text"]}) for d in data]
document_store.write_documents(docs, policy="overwrite")

# Set parameters
model = "llama3.1:8b"
temperature = 0
top_k = 6
retriever_type = "bm25"

llm = OllamaGenerator(
    model=model,
    url="http://localhost:11434",
    generation_kwargs={"temperature": temperature},
)


# Create results file for the experiment
dt = str(datetime.now()).split(".")[0].replace(" ", "_").replace(":", "-")
output_filename = f"../output/rener_{dt}.json"
results = {
    "model": model,
    "temperature": temperature,
    "retriever_type": retriever_type,
}

for d in range(len(input_data)):
    start_time = time()

    pbar = tqdm(input_data)
    for item in pbar:
        # Define query
        query = f"Input: {item['text']}\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"

        # Run retrieval
        if retriever_type == "bm25":
            retriever = InMemoryBM25Retriever(document_store=document_store)
            retrieval_result = retriever.run(query=query, top_k=top_k)
            retrieved_docs = retrieval_result["documents"]
        elif retriever_type == "sentence_similarity":
            ranker = SentenceTransformersSimilarityRanker()
            ranker.warm_up()
            retrieval_result = ranker.run(query=query, documents=docs, top_k=top_k)
            retrieved_docs = retrieval_result["documents"]

        # Format retrieved docs using llama's prompt format
        context = "".join(
            [
                f"Input: {doc.content}\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nOutput: {doc.meta['output']}<|eot_id|>\n\n"
                for i, doc in enumerate(retrieved_docs)
            ]
        ).strip()

        # Build prompt
        prompt = guidelines_llama3_legal_lite + default_examples + context + query

        # Run the LLM and print output
        generation_result = llm.run(
            prompt=prompt,
            generation_kwargs={"temperature": temperature},
        )
        llm_output = generation_result["replies"][0]

        item["pred"] = llm_output.replace("Output: \n", "")

        pred = extract_ner_labels(item["pred"])

        # Apply preprocessing to account for LLMs' rewriting
        pred = [(w[0].strip().title(), w[1]) for w in pred]

        # Set annotations in output dict
        item["annotations"] = pred

        # print("--- Context ---")
        # print(context)
        # print()
        # print("--- LLM I/O ---")
        # print(query)
        # print(llm_output)
        # print()
        # print("---")
        # print()

    tot_runtime = time() - start_time


with open(output_filename, "w") as file:
    json.dump(results, file)


In [ ]:
# Save as jsonl
with open("../data/pred/docs_parallel.jsonl", "w") as output_file:
    for entry in input_data:
        json.dump(entry, output_file)
        output_file.write("\n")